In [ ]:
import torch
from diffusers import FluxTransformer2DModel, FluxPipeline
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBitsAndBytesConfig

# Quantize the transformer to NF4 while loading from the original bf16 weights.
transformer_quant_config = DiffusersBitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)
transformer = FluxTransformer2DModel.from_pretrained(
    "UDream/flux2-dev-turbo-Q4_K_M",
    subfolder="transformer",
    quantization_config=transformer_quant_config,
    torch_dtype=torch.bfloat16,
)

# Same for the T5 text encoder (~9.5GB in bf16) — the other big memory hog.
text_encoder_quant_config = TransformersBitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)
text_encoder_2 = T5EncoderModel.from_pretrained(
    "UDream/flux2-dev-turbo-Q4_K_M",
    subfolder="text_encoder_2",
    quantization_config=text_encoder_quant_config,
    torch_dtype=torch.bfloat16,
)

pipe = FluxPipeline.from_pretrained(
    "UDream/flux2-dev-turbo-Q4_K_M",
    transformer=transformer,
    text_encoder_2=text_encoder_2,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()

prompt = "A realistic, high quality pitbull mixed breed dog flying in a Spiderman suite. The suit covers all the body of the dog, and only the head is uncovered. The sky is blue with a few white clouds but the moon is visible."
image = pipe(
    prompt,
    height=512,
    width=512,
    guidance_scale=3.5,
    num_inference_steps=9,
    max_sequence_length=512,
    generator=torch.Generator("cpu").manual_seed(0)
).images[0]
image.save("flux-dev.png")


Loading weights: 100%|██████████| 219/219 [00:04<00:00, 44.54it/s]
c:\Users\esteb\miniconda3\envs\z-image\Lib\site-packages\diffusers\utils\deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
100%|██████████| 9/9 [00:08<00:00,  1.10it/s]
